In [ ]:
# Boilerplate imports

%load_ext autoreload
%autoreload 2
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

import numpy as np
np.set_printoptions(linewidth=5000)
import pandas as pd
pd.set_option("display.max_rows", 2000)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 2000)
from scipy.stats import norm
import scipy

from bokeh.plotting import figure
from bokeh.io import show, output_notebook, push_notebook
from bokeh.layouts import gridplot, row, column
from bokeh.models import (BasicTicker, ColorBar, ColumnDataSource, LinearColorMapper, PrintfTickFormatter, HoverTool)
from bokeh.transform import transform
from bokeh.models.formatters import DatetimeTickFormatter
output_notebook()

from functools import partial

import logging as Log
Log.basicConfig(level=Log.INFO)

verbose = False

In [ ]:
# Pricing and IV

# Black-Scholes

def bsprice(S, K, vol, cp, ct, vt, box, roll):
    F = S * np.exp(roll * ct)
    df = np.exp(-box * ct) 
    stddev = vol * np.sqrt(vt)
    d1 = np.log(F/K)/stddev + stddev/2.0
    d2 = d1 - stddev
    price = cp * df * (F * norm.cdf(cp*d1) - K * norm.cdf(cp*d2))    
    return price

def black_scholes_iv(option_price, S, K, cp, ct, vt, box, roll, min_vol=0.001, max_vol=10):
    def func(vol):
        return bsprice(S, K, vol, cp, ct, vt, box, roll) - option_price
    if func(min_vol) * func(max_vol) > 0:
        result = np.nan
    else:
        result = scipy.optimize.brentq(func, min_vol, max_vol)
    return result

# Bachelier

def bachelier_price(S, K, vol, cp, ct, vt, box, roll):
    F = S * np.exp(roll * ct)
    df = np.exp(-box * ct) 
    stddev = vol * np.sqrt(vt)
    d = cp * (F - K) / stddev
    price = df * (cp * (F - K) * norm.cdf(d) + stddev * norm.pdf(d))
    return price

def bachelier_iv(option_price, S, K, cp, ct, vt, box, roll, min_vol=0.001, max_vol=10):
    def func(vol):
        return bachelier_price(S, K, vol*S, cp, ct, vt, box, roll) - option_price
    if func(min_vol) * func(max_vol) > 0:
        result = np.nan
    else:
        result = scipy.optimize.brentq(func, min_vol, max_vol)
    return result

# Plotting 

def plot_vs_strike(strikes, ylabel, data):
    '''data is a list of [name, color, ary]'''
    p = figure(plot_width=600, plot_height=400, x_axis_label='Strike', y_axis_label=ylabel)
    for which in data:
        name, color, ary = which
        p.circle(strikes, ary, color=color, legend_label=name)
        p.line(strikes, ary, color=color, legend_label=name)
    p.legend.location = "top_right"
    p.legend.click_policy="hide"
    return p

In [ ]:
# Set up parameters for our scenario, with zero rates for simplicity
#ct, vt, atfvol, roll, box, S = 0.077055, 0.080295, 0.209555, 0.005973, 0.004597, 150.55
ct, vt, atfvol, roll, box, S = 0.077055, 0.080295, 0.209555, 0.00, 0.0, 150.55

# Suppose "truth" is constant normal vol, compute B-S IVs for calls
vol = atfvol
Log.info("Using vol: %s", vol)

cp = 1

strikes = np.arange(95.0, 225.0, 5.0)
call_prices_normal = np.array([bachelier_price(S, K, vol*S, cp, ct, vt, box, roll) for K in strikes])
call_prices_bs = np.array([bsprice(S, K, vol, cp, ct, vt, box, roll) for K in strikes])

implied_vols_bs = np.array([black_scholes_iv(option_price, S, K, cp, ct, vt, box, roll) for K, option_price in zip(strikes, call_prices_normal)])
implied_vols_normal = np.array([bachelier_iv(option_price, S, K, cp, ct, vt, box, roll) for K, option_price in zip(strikes, call_prices_normal)])

call_price_plot = plot_vs_strike(strikes, 'Call Price',  [('Normal call price', 'red', call_prices_normal), ('Lognormal call price', 'green', call_prices_bs)])
iv_plot = plot_vs_strike(strikes, 'IV', [('B-S IV', 'red', implied_vols_bs), ('Normal IV', 'green', implied_vols_normal)])
show(row(call_price_plot, iv_plot))


In [ ]:
# ATM
bachelier_price(S, S*np.exp(roll*ct), vol*S, cp, ct, vt, box, roll), bsprice(S, S*np.exp(roll*ct), vol, cp, ct, vt, box, roll)

In [ ]:
# ITM
bachelier_price(S, S*np.exp(roll*ct) * 0.9, vol*S, cp, ct, vt, box, roll), bsprice(S, S*np.exp(roll*ct) * 0.9, vol, cp, ct, vt, box, roll)

In [ ]:
# OTM
bachelier_price(S, S*np.exp(roll*ct) * 1.1, vol*S, cp, ct, vt, box, roll), bsprice(S, S*np.exp(roll*ct) * 1.1, vol, cp, ct, vt, box, roll)